**Note:** This is a DDL notebook. Run this only once

In [1]:
from pyspark.sql import SparkSession

from seed.nessie import conf

spark: SparkSession = SparkSession.builder.config(conf=conf).getOrCreate()
print(f"Spark {spark.version} is up and running!")

25/07/28 22:59:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/07/28 22:59:26 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 3.5.5 is up and running!


In [2]:
# Create a namespace in spark_catalog catalog

# spark.sql("DROP NAMESPACE IF EXISTS dev;")

spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.dev;")

spark.sql("SHOW NAMESPACES FROM nessie").show()

+---------+
|namespace|
+---------+
|      dev|
+---------+



In [3]:
# Read flights data from parquet file

df = spark.read.parquet("s3a://seed/flights-1m.parquet")
df.show(5)

df.printSchema()

df.createOrReplaceTempView("raw_flights")

spark.sql("""
    SELECT
        MIN(FL_DATE) AS min_date,
        MAX(FL_DATE) AS max_date,
        COUNT(*) AS num_rows
    FROM raw_flights;
""").show()

spark.sql("""
    SELECT
        FL_DATE,
        COUNT(*) AS num_rows
    FROM raw_flights
    GROUP BY FL_DATE
    ORDER BY FL_DATE
    LIMIT 10;
""").show()

+----------+---------+---------+--------+--------+---------+---------+
|   FL_DATE|DEP_DELAY|ARR_DELAY|AIR_TIME|DISTANCE| DEP_TIME| ARR_TIME|
+----------+---------+---------+--------+--------+---------+---------+
|2006-01-01|        5|       19|     350|    2475| 9.083333|12.483334|
|2006-01-02|      167|      216|     343|    2475|11.783334|15.766666|
|2006-01-03|       -7|       -2|     344|    2475| 8.883333|12.133333|
|2006-01-04|       -5|      -13|     331|    2475| 8.916667|    11.95|
|2006-01-05|       -3|      -17|     321|    2475|     8.95|11.883333|
+----------+---------+---------+--------+--------+---------+---------+
only showing top 5 rows

root
 |-- FL_DATE: date (nullable = true)
 |-- DEP_DELAY: short (nullable = true)
 |-- ARR_DELAY: short (nullable = true)
 |-- AIR_TIME: short (nullable = true)
 |-- DISTANCE: short (nullable = true)
 |-- DEP_TIME: float (nullable = true)
 |-- ARR_TIME: float (nullable = true)



+----------+----------+--------+
|  min_date|  max_date|num_rows|
+----------+----------+--------+
|2006-01-01|2006-02-28| 1000000|
+----------+----------+--------+

+----------+--------+
|   FL_DATE|num_rows|
+----------+--------+
|2006-01-01|   17618|
|2006-01-02|   19156|
|2006-01-03|   19290|
|2006-01-04|   18869|
|2006-01-05|   19534|
|2006-01-06|   19553|
|2006-01-07|   16236|
|2006-01-08|   18506|
|2006-01-09|   19483|
|2006-01-10|   18541|
+----------+--------+



In [4]:
# Create flights table from parquet file

spark.sql("""
    CREATE TABLE IF NOT EXISTS nessie.dev.flights 
    USING iceberg
    PARTITIONED BY (fl_date)
    TBLPROPERTIES ('gc.enabled' = 'true')
    AS
    SELECT
        *
    FROM raw_flights;
""")

spark.sql("SELECT * FROM nessie.dev.flights LIMIT 5;").show()

# spark.sql("DROP TABLE nessie.dev.flights PURGE;")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
25/07/28 23:00:02 WARN NessieUtil: The Iceberg property 'gc.enabled' and/or 'write.metadata.delete-after-commit.enabled' is enabled on table 'dev.flights' in NessieCatalog. This will likely make data in other Nessie branches and tags and in earlier, historical Nessie commits inaccessible. The recommended setting for those properties is 'false'. Use the 'nessie-gc' tool for Nessie reference-aware garbage collection.


+----------+---------+---------+--------+--------+---------+---------+
|   FL_DATE|DEP_DELAY|ARR_DELAY|AIR_TIME|DISTANCE| DEP_TIME| ARR_TIME|
+----------+---------+---------+--------+--------+---------+---------+
|2006-02-14|        0|       -6|     335|    2475|      9.0|12.066667|
|2006-02-14|       -4|      -24|     270|    2475| 9.433333|    17.35|
|2006-02-14|       -4|        9|     338|    2475|11.933333|     15.2|
|2006-02-14|       55|       60|     281|    2475|13.416667|21.816668|
|2006-02-14|       -3|      -11|     490|    3784|10.033334|     14.5|
+----------+---------+---------+--------+--------+---------+---------+



In [5]:
spark.stop()